
# NB-A — Inference + Feedback (Gradio) — ClearML Community
*Projet : `Image-Classifier/Gradio-Retrain-Demo`*

Ce notebook lance une app **Gradio** pour classer des images CIFAR-10 (10 classes)
et **collecter du feedback utilisateur** (labels confirmés/corrigés).  
Toutes les 5 images confirmées, on **publie** le `Dataset("image-feedback")`
avec le tag `auto-feedback` pour déclencher la **pipeline de réentraînement**.

> ⚠️ Prérequis : un serveur ClearML (cloud public ou self-hosted) et `clearml-agent` actif côté NB-D ou autre machine.


In [3]:
!pip -q install clearml gradio

In [ ]:
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
# tp3
%env CLEARML_API_ACCESS_KEY=VES7TTVL4QDF36XPHCPXLAV1KDIZY4
%env CLEARML_API_SECRET_KEY=ygijnEsVvuMODXfwaXay-EIcKZbQVReuv18Bd9O5qskRWuHsAbn_V0w1b5waRQbgaNE

env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml
env: CLEARML_FILES_HOST=https://files.clear.ml
env: CLEARML_API_ACCESS_KEY=1S42Z803M2MAO8JA8INCFSLP2PJI7X
env: CLEARML_API_SECRET_KEY=m5-KPmM1h6xbCfxX0oqpYwVO2wGeMbTzDXoUIIJQT8q3rPBHAg8SnST9OtprSbKmY68


In [ ]:
import os, io, json, time, uuid, traceback, shutil
from pathlib import Path
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image

import gradio as gr
from clearml import Model, Dataset

# -------------------- CONFIG --------------------
PROJECT = "Image-Classifier/Gradio-Retrain-Demo-IABD"
DATASET_NAME = "image-feedback"
# Use a writable local directory instead of /content, which is only valid in Colab.
BASE_DIR = Path.home() / "clearml_feedback"
PENDING_DIR = BASE_DIR / "pending"
UPLOADED_DIR = BASE_DIR / "uploaded"
STATE_FILE = BASE_DIR / "state.json"    # to count stored samples

BASE_DIR.mkdir(parents=True, exist_ok=True)
PENDING_DIR.mkdir(parents=True, exist_ok=True)
UPLOADED_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- MODEL LOADING --------------------
def build_resnet18(num_classes=10):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

CLASSES = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_production_model() -> nn.Module:
    """Load last published model tagged 'production'; fallback to random init."""
    try:
        prods = Model.query_models(project_name=PROJECT, tags=["candidate"], only_published=True) or []
        if not prods:
            print("⚠️ No PRODUCTION model found; using randomly initialized ResNet18.")
            m = build_resnet18(10).to(DEVICE); m.eval(); return m
        prod = prods[0]
        local = prod.get_local_copy()
        print("✅ Loaded production model:", prod.id, "from", local)
        m = build_resnet18(10).to(DEVICE)
        sd = torch.load(local, map_location=DEVICE)
        m.load_state_dict(sd, strict=False)
        m.eval()
        return m
    except Exception:
        print("⚠️ Failed to load production model, using random init:\n", traceback.format_exc())
        m = build_resnet18(10).to(DEVICE); m.eval(); return m

model = load_production_model()

# -------------------- PREPROCESS --------------------
TFM = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

@torch.no_grad()
def predict_image(pil: Image.Image) -> Tuple[str, float]:
    pil = pil.convert("RGB")
    x = TFM(pil).unsqueeze(0).to(DEVICE)  # [1,3,32,32]
    logits = model(x)
    prob = torch.softmax(logits, dim=1)[0]
    conf, idx = torch.max(prob, dim=0)
    return CLASSES[int(idx)], float(conf.item())


⚠️ No PRODUCTION model found; using randomly initialized ResNet18.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>